# Graph Convolutional Network (GCN) Example

This notebook demonstrates a simple Graph Convolutional Network (GCN) implemented using PyTorch. The GCN will learn to transform node features based on the graph structure.

## 1. Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as o

## 2. Define Input Data

We define node features `X` and an adjacency matrix `A` representing the graph structure.

In [2]:
# Node features (3 nodes, 2 features per node)
X = torch.tensor(
    [[1., 2.],
     [2., 3.],
     [4., 1.]]
)

# Adjacency matrix (3x3 representing connections between nodes)
A = torch.tensor(
    [[0., 1., 1.],
     [1., 0., 1.],
     [1., 1., 0.]]
)

## 3. Pre-processing the Adjacency Matrix

To ensure stable training and capture self-connections, we add self-loops to the adjacency matrix and then normalize it. The normalization is done using the degree matrix `D`.

In [3]:
# Add self-loops to the adjacency matrix
I = torch.eye(A.shape[0])
A_hat = A + I

# Calculate the degree matrix (D) and its inverse square root (D_inv_sqrt)
D = torch.diag(torch.sum(A_hat, dim=1))
D_inv_sqrt = torch.linalg.inv(torch.sqrt(D))

# Normalize the adjacency matrix
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt

## 4. Define the GCN Model

The GCN layer involves a matrix multiplication with the normalized adjacency matrix, followed by a linear transformation and an activation function (ReLU in this case).

In [4]:
class GCN(nn.Module):

    def __init__(self):
        super().__init__()
        # Linear layer to transform node features (input features: 2, output features: 2)
        self.W = nn.Linear(2, 2)

    def forward(self, X):
        # Apply graph convolution: H = A_norm @ X
        H = A_norm @ X
        # Apply linear transformation: H = H @ W
        H = self.W(H)
        # Apply ReLU activation
        H = torch.relu(H)

        return H

## 5. Instantiate Model and Optimizer

We create an instance of our `GCN` model and define an Adam optimizer for training.

In [5]:
model = GCN()

# Define the optimizer (Adam) with a learning rate of 0.01
optimizer = o.Adam(model.parameters(), lr=0.01)

## 6. Define Target Output

This is the desired output `Y` that our GCN model should try to predict.

In [6]:
# Target output for each node
Y = torch.tensor(
    [[3., 3.],
     [3., 3.],
     [3., 3.]]
)

## 7. Training Loop

The model is trained for 1000 epochs. In each epoch, it calculates the prediction, computes the mean squared error loss, performs backpropagation, and updates the model's weights.

In [7]:
print("Starting training...")
for epoch in range(1000):
    # Forward pass to get predictions
    prediction = model(X)

    # Calculate Mean Squared Error loss
    loss = torch.mean((prediction - Y) ** 2)

    # Zero gradients, perform backward pass, and update weights
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

print("Training complete.")

Training complete.


## 8. Final Prediction

After training, we can observe the model's final output for the input features `X`.

In [8]:
print("\nFinal Prediction:")
print(model(X))


Final Prediction:
tensor([[3., 0.],
        [3., 0.],
        [3., 0.]], grad_fn=<ReluBackward0>)
